EGM722 Programming for GIS and Remote Sensing Assessment 2

Title: GIS Assessment of Geothermal Energy Potential and Environmental Constraints in Northern Ireland.

Aim: This project compares potentially suitable geothermal areas with environmentally protected areas in Northern Ireland.

1 Import Python modules

In [ ]:
import sys

!{sys.executable} -m pip install geopandas folium contextily

2 Import Python libraries

In [ ]:
# File and folder handling
from pathlib import Path
import os
import zipfile
import urllib.request

# Data analysis
import pandas as pd

# Spatial data handling
import geopandas as gpd

# Mapping and plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import contextily as ctx

# Interactive web mapping
import folium

# Spatial geometry operations
from shapely.geometry import box

# Suppress warnings for cleaner notebook output
import warnings
warnings.filterwarnings("ignore")

# Display all dataframe columns in notebook outputs
pd.set_option("display.max_columns", 100)

3 Set file paths

This section defines the folder structure used throughout the project.

Separate folders are used for:
- raw downloaded datasets,
- processed GIS outputs,
- and exported figures/results.

Using a consistent folder structure improves reproducibility and organisation of spatial analysis workflows. The folders are automatically created if they do not already exist.

The project uses pathlib.Path because it provides a platform-independent and reliable method for handling file paths in Python.

In [ ]:
from pathlib import Path

# Project folders
PROJECT_DIR = Path.cwd().parent

DATA_RAW = PROJECT_DIR / "data_raw"
DATA_PROCESSED = PROJECT_DIR / "data_processed"
OUTPUTS = PROJECT_DIR / "outputs"

# Create folders if needed
for folder in [DATA_RAW, DATA_PROCESSED, OUTPUTS]:
    folder.mkdir(exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data folder:", DATA_RAW)
print("Processed data folder:", DATA_PROCESSED)
print("Outputs folder:", OUTPUTS)

4 Set Coordinate Reference Systems (CRS)

In [ ]:
# British National Grid is used by the BGS deep geothermal shapefile
BNG_CRS = "EPSG:27700"

# Irish National Grid is useful for NI environmental datasets
ING_CRS = "EPSG:29903"

# WGS84 is required for Folium web maps
WEB_CRS = "EPSG:4326"

# Use BNG as the analysis CRS because the BGS geothermal layer is supplied in BNG
TARGET_CRS = BNG_CRS

5 Define resusable functions

In [ ]:
def download_and_extract(url, zip_path, extract_folder):
    """
    Download a zipped GIS dataset and extract it locally.

    Parameters
    ----------
    url : str
        Direct URL to the zipped dataset.
    zip_path : pathlib.Path
        Local path where the ZIP file will be saved.
    extract_folder : pathlib.Path
        Folder where the ZIP file will be extracted.

    Returns
    -------
    pathlib.Path
        Path to the extracted folder.
    """
    if not zip_path.exists():
        print(f"Downloading: {url}")
        urllib.request.urlretrieve(url, zip_path)
    else:
        print(f"Already downloaded: {zip_path.name}")

    extract_folder.mkdir(exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_folder)

    print(f"Extracted to: {extract_folder}")
    return extract_folder

def find_file(folder, extension):
    """
    Find the first file with a chosen extension inside a folder.

    Parameters
    ----------
    folder : pathlib.Path
        Folder to search.
    extension : str
        File extension pattern, for example '*.shp' or '*.gpkg'.

    Returns
    -------
    pathlib.Path
        Path to the first matching file.
    """
    files = list(folder.rglob(extension))

    if len(files) == 0:
        raise FileNotFoundError(f"No {extension} file found in {folder}")

    return files[0]

def load_vector(path, target_crs=TARGET_CRS, layer=None):
    """
    Load a vector GIS dataset and reproject it to a target CRS.

    Parameters
    ----------
    path : str or pathlib.Path
        Path to the GIS file, for example SHP, GeoJSON or GeoPackage.
    target_crs : str
        Coordinate reference system to reproject the dataset into.
    layer : str, optional
        Layer name to read when using a GeoPackage with multiple layers.

    Returns
    -------
    geopandas.GeoDataFrame
        Loaded and reprojected spatial dataset.
    """
    # Read the spatial file using GeoPandas.
    gdf = gpd.read_file(path, layer=layer)

    # Stop the workflow if the dataset has no CRS.
    if gdf.crs is None:
        raise ValueError(f"{path} has no CRS defined.")

    # Reproject to the chosen CRS so all datasets align spatially.
    return gdf.to_crs(target_crs)

def clean_geodataframe(gdf):
    """
    Clean a GeoDataFrame by removing missing geometries and repairing invalid ones.

    Parameters
    ----------
    gdf : geopandas.GeoDataFrame
        Input spatial dataset.

    Returns
    -------
    geopandas.GeoDataFrame
        Cleaned spatial dataset.
    """
    # Work on a copy so the original data is not modified accidentally.
    gdf = gdf.copy()

    # Remove rows without valid geometry.
    gdf = gdf[gdf.geometry.notna()]
    gdf = gdf[~gdf.geometry.is_empty]

    # Repair invalid polygon geometries using a zero-width buffer.
    gdf["geometry"] = gdf.geometry.buffer(0)

    return gdf

def classify_geothermal_play(row):
    """
    Assign an indicative geothermal suitability score using BGS geothermal play attributes.

    The scoring is based on general geothermal reasoning:
    sedimentary basins, hydrothermal systems, and fault/fracture-controlled settings
    are considered more favourable than poorly defined or low-permeability settings.

    Parameters
    ----------
    row : pandas.Series
        One row from the geothermal GeoDataFrame.

    Returns
    -------
    int
        Suitability score from 1 to 5, where 5 is highest.
    """
    text = " ".join([
        str(row.get("Geotherm_2", "")),
        str(row.get("GeologicHa", "")),
        str(row.get("GeologicCo", "")),
        str(row.get("BGSinforma", "")),
        str(row.get("Status", ""))
    ]).lower()

    score = 1

    # Sedimentary basins are often favourable for hydrothermal geothermal resources
    if "sedimentary" in text or "basin" in text:
        score += 2

    # Hydrothermal systems are directly relevant to geothermal production
    if "hydrothermal" in text:
        score += 1

    # Faults and fractures can increase permeability
    if "fault" in text or "fracture" in text:
        score += 1

    # Sandstone and limestone can be favourable aquifer lithologies
    if "sandstone" in text or "limestone" in text:
        score += 1

    # Granites may be relevant for deep geothermal heat but often need fracture permeability
    if "granite" in text:
        score += 1

    return min(score, 5)

def classify_score(score):
    """
    Convert a numeric score into a class.

    Parameters
    ----------
    score : float
        Suitability or sensitivity score.

    Returns
    -------
    str
        Class label.
    """
    if score >= 4:
        return "High"
    elif score >= 3:
        return "Moderate"
    elif score >= 2:
        return "Low"
    else:
        return "Very low"

def add_north_arrow(ax, x=0.95, y=0.15):
    """
    Add a simple north arrow to a Matplotlib map.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Map axis to draw the north arrow on.
    x : float
        X position in axis coordinates.
    y : float
        Y position in axis coordinates.

    Returns
    -------
    None
    """
    ax.annotate(
        "N",
        xy=(x, y + 0.08),
        xytext=(x, y),
        xycoords="axes fraction",
        ha="center",
        va="center",
        fontsize=14,
        fontweight="bold",
        arrowprops=dict(
            facecolor="black",
            width=4,
            headwidth=10
        )
    )

6 Download and extract data

6.1 Store the web address for downloading Geothermal data

In [ ]:
GEOTHERMAL_URL = "https://webservices.bgs.ac.uk/accessions/download/185538?fileName=Deep_Geothermal_Areas_v2.zip"

6.2 Download and extract Geothermal data

In [ ]:
GEOTHERMAL_FOLDER = download_and_extract(
    GEOTHERMAL_URL,
    DATA_RAW / "Deep_Geothermal_Areas_v2.zip",
    DATA_RAW / "deep_geothermal_areas_v2"
)

# List extracted files
for file in GEOTHERMAL_FOLDER.rglob("*"):
    print(file)

6.3 Locate the Geothermal shapefile from the zip file

In [ ]:
# Use recursive global search and wildcard to search through the folder 
# to find all files ending with .shp, i.e., shapefiles
# returns the results as a list
shp_files = list(GEOTHERMAL_FOLDER.rglob("*.shp"))

print("Shapefiles found:")

# for loop prints the file path for the shapefiles found in the folder
for file in shp_files:
    print(file)

# Select the first shapefile in the list (expect there to be only one)
GEOTHERMAL_FILE = shp_files[0]

# Print the shapefile 
print("Using:", GEOTHERMAL_FILE)

6.4 Load Geothermal data into GeoPanda DataFrame

In [ ]:
# Find shapefile in the extracted dataset folder using find_file function
GEOTHERMAL_FILE = find_file(GEOTHERMAL_FOLDER, "*.shp")

# Load the shapefile into GeoPandas using load_vector function
geothermal = load_vector(GEOTHERMAL_FILE)

# Clean the dataset using the clean_geodataframe function
geothermal = clean_geodataframe(geothermal)

# Print the number of features (rows) in the dataset
print("Geothermal records:", len(geothermal))

# Display the first 5 rows of the 'geothermal' GeoDataFrame
display(geothermal.head())

# Print all column names in the dataset
print(geothermal.columns)

6.5 Score geothermal suitability

In [ ]:
# Create a new column in the GeoDataFrame called 'geothermal score'
# Run the classify_geothermal_play() function on every row in the GeoDataFrame
geothermal["geothermal_score"] = geothermal.apply(
    classify_geothermal_play,
    axis=1 # apply function row by row instead of column by column
)

# Create another new column called 'geothermal class'
# Convert numeric scores into classed categories, e.g., low, moderate, and high
# Take each geothermal score and pass it into classify_score() function
geothermal["geothermal_class"] = geothermal["geothermal_score"].apply(
    classify_score
)

# Display selected columns in a formatted table to inspect outputs
display(
    geothermal[
        [
            "Geotherm_2",
            "GeologicHa",
            "GeologicCo",
            "BGSinforma",
            "geothermal_score",
            "geothermal_class"
        ]
    ].head()
)

6.6 Store the web addresses for downloading protected area datasets

In [ ]:
ASSI_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/ASSI%20-%20Irish%20National%20Grid_5.zip"
SAC_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/Special%20Areas%20of%20Conservation%20-%20Irish%20National%20Grid_1.zip"
SPA_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/Special%20Protected%20Areas%20-%20Irish%20National%20Grid_1.zip"
RAMSAR_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/Ramsar%20Sites%20-%20Irish%20National%20Grid_1.zip"
AONB_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/AONB%20-%20Irish%20National%20Grid_0.zip"
NNR_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/National%20Nature%20Reserves%20-%20Irish%20National%20Grid_1.zip"
WHS_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/World_Heritage_Site%20-%20Irish%20National%20Grid.zip"

6.7 Download and extract protected area datasets

In [ ]:
# Download and extract datasets
ASSI_FOLDER = download_and_extract(
    ASSI_URL, 
    DATA_RAW / "assi.zip", 
    DATA_RAW / "assi"
)

SAC_FOLDER = download_and_extract(
    SAC_URL, 
    DATA_RAW / "sac.zip", 
    DATA_RAW / "sac"
)

SPA_FOLDER = download_and_extract(
    SPA_URL, 
    DATA_RAW / "spa.zip", 
    DATA_RAW / "spa"
)

RAMSAR_FOLDER = download_and_extract(
    RAMSAR_URL, 
    DATA_RAW / "ramsar.zip", 
    DATA_RAW / "ramsar"
)

AONB_FOLDER = download_and_extract(
    AONB_URL, 
    DATA_RAW / "aonb.zip", 
    DATA_RAW / "aonb"
)

NNR_FOLDER = download_and_extract(
    NNR_URL, 
    DATA_RAW / "nnr.zip", 
    DATA_RAW / "nnr"
)

WHS_FOLDER = download_and_extract(
    WHS_URL, 
    DATA_RAW / "whs.zip", 
    DATA_RAW / "whs"
)

# Load shapefiles using the load_vector() function and the find_file() function
# find-file() function searches through the extracted folder and finds the shapefile
# load_vector() function loads the shapefile into GeoPandas and reprojects it
# Creates GeoDataFrames
assi = load_vector(find_file(ASSI_FOLDER, "*.shp"))
sac = load_vector(find_file(SAC_FOLDER, "*.shp"))
spa = load_vector(find_file(SPA_FOLDER, "*.shp"))
ramsar = load_vector(find_file(RAMSAR_FOLDER, "*.shp"))
aonb = load_vector(find_file(AONB_FOLDER, "*.shp"))
nnr = load_vector(find_file(NNR_FOLDER, "*.shp"))
whs = load_vector(find_file(WHS_FOLDER, "*.shp"))

# Clean and repair geometry using the clean_geodataframe() function
assi = clean_geodataframe(assi)
sac = clean_geodataframe(sac)
spa = clean_geodataframe(spa)
ramsar = clean_geodataframe(ramsar)
aonb = clean_geodataframe(aonb)
nnr = clean_geodataframe(nnr)
whs = clean_geodataframe(whs)

# Inspect GeoDataFrames by printing the number of polygons / features in the datasets
print("ASSI records:", len(assi))
print("SAC records:", len(sac))
print("SPA records:", len(spa))
print("Ramsar records:", len(ramsar))
print("AONB records:", len(aonb))
print("NNR records:", len(nnr))
print("WHS records:", len(whs))

6.8 Create environmental sensitivity scores

In [ ]:
# Scores assigned based on assumed environmental planning sensitivity
# Higher scores indicate stronger environmental constraint
# SAC, SPA, Ramsar, NNR and WHS are assigned the maximum score because they represent internationally or European designated protected areas
# ASSI is scored slightly lower due to their national ecological and geological importance
# AONB is scored lower again because it is mainly a landscape designation

assi["constraint_type"] = "ASSI"
assi["environment_score"] = 4

sac["constraint_type"] = "SAC"
sac["environment_score"] = 5

spa["constraint_type"] = "SPA"
spa["environment_score"] = 5

ramsar["constraint_type"] = "Ramsar"
ramsar["environment_score"] = 5

aonb["constraint_type"] = "AONB"
aonb["environment_score"] = 3

nnr["constraint_type"] = "National Nature Reserve"
nnr["environment_score"] = 5

whs["constraint_type"] = "World Heritage Site"
whs["environment_score"] = 5

# Merge all protected-area layers into one combined constraints layer
protected_areas = pd.concat(
    [assi, sac, spa, ramsar, aonb, nnr, whs],
    ignore_index=True
)

# Convert the merged DataFrame back into a GeoDataFrame
protected_areas = gpd.GeoDataFrame(
    protected_areas,
    geometry="geometry",
    crs=TARGET_CRS
)

# Clean and repair geometries before spatial overlay
protected_areas = clean_geodataframe(protected_areas)

# Display the number of rows / features in the GeoDataFrame to check merge executed correctly
print("Combined protected-area records:", len(protected_areas))

# Display the first 5 rows of the GeoDataFrame in a formatted table  to inspect the columns / 
# attributes and the environmental sensitivity score assignment
display(protected_areas.head())

6.9 Create Northern Ireland study area boundary

In [ ]:
from shapely.geometry import box

xmin, ymin, xmax, ymax = ni_boundary.total_bounds

ni_extent = gpd.GeoDataFrame(
    geometry=[box(xmin, ymin, xmax, ymax)],
    crs=TARGET_CRS
)

6.10 Clip geothermal and protected areas data to the extent of the study area

In [ ]:
geothermal_ni = gpd.clip(geothermal, ni_extent)
protected_areas_ni = gpd.clip(protected_areas, ni_extent)

print("Geothermal records after clipping:", len(geothermal_ni))
print("Protected-area records after clipping:", len(protected_areas_ni))

7 Analysis - Overlay geothermal areas with protected areas

In [ ]:
# Identify where geothermal areas overlap with environmentally protected areas
# Creates a new GeoDataFrame called 'conflict_areas' which contains only the areas where both datasets overlap
conflict_areas = gpd.overlay(
    geothermal_ni,
    protected_areas_ni,
    how="intersection" # intersection keeps areas where the polygons overlap
)

# Calculate area of 'conflict_areas' in square kilometres
# Creates a new column 'area_km2' to store polygon area
conflict_areas["area_km2"] = conflict_areas.geometry.area / 1_000_000

# Print the number of overapping polygons created in 'conflict_areas' GeoDataFrame
# Check that the overlay worked, the intersection exists, and that the data loaded properly
print("Overlap records:", len(conflict_areas))

# Display the first 5 rows of the overlap datset in an output table
# Inspect the overlap attributes, scores, geometry, and calculated area
display(conflict_areas.head())

8 Classify planning conflict

In [ ]:
# Define a classification function 
# Function examines one row at a ttime, compares scores, and returns a text interpretation
def classify_conflict(row):
    """
    Classify overlap between geothermal potential and environmental sensitivity

    Parameters
    ----------
    row : pandas.Series
        Row containing geothermal and environmental scores

    Returns
    -------
    str
        Conflict interpretation category
    """
    geothermal_score = row["geothermal_score"] # read geothermal score from current row
    environment_score = row["environment_score"] # read environmental sensitivity score from the row

    # Decision rules
    # Rule 1
    if geothermal_score >= 4 and environment_score >= 4:
        return "High geothermal potential / high environmental constraint" 
    # if geothermal potential is high and environmental sensitivity is high then the statement 'high geothermal potential / high environmental constraint' will be returned
    # therefore, there is a potential planning conflict and constraint, and any geothermal project would require detailed environmental impact assessment
    # Rule 2
    elif geothermal_score >= 4 and environment_score < 4:
        return "High geothermal potential / lower environmental constraint"
    # potentially favourable devleopment areas with lower conflict zones
    # Rule 3
    elif geothermal_score >= 3 and environment_score >= 4:
        return "Moderate geothermal potential / high environmental constraint"
    # moderate geothermal opportunity but environmentally sensitive
    # therefore, geothermal project opportunities may be constrained, suitability is uncertain, and cautious planning is required
    # else statement
    else:
        return "Lower priority or lower conflict"
    # applies if none of the earlier conditions are met
    # indicates an area of lower geothermal suitability and lower environmental importance

# classify_conflict(row) function applied to each row
# Creates a new column called 'conflict_class'
conflict_areas["conflict_class"] = conflict_areas.apply(
    classify_conflict,
    axis=1 # runs the function on every row
)

# Display the first 5 rows of the selected columns in a formatted table to allow for inspection of results
display(
    conflict_areas[
        ["geothermal_class", "constraint_type", "environment_score",
         "conflict_class", "area_km2"]
    ].head()
)

9 Results table

In [ ]:
# Create a DataFrame called 'summary_table' to contain grouped statistics and summarised GIS results
summary_table = (
    conflict_areas # select input data
    .groupby(["conflict_class", "constraint_type"]) # group data into categories
    .agg( # aggregate summary statistics for each group
        feature_count=("conflict_class", "count"), # count the number of polygons in each category
        total_area_km2=("area_km2", "sum"), # sum overlapping polygon areas
        mean_geothermal_score=("geothermal_score", "mean"), # calculate average geothermal suitability
        mean_environment_score=("environment_score", "mean") # calculate average environmental sensitivity
    )
    .reset_index() # convert grouped categories back into normal columns
)

# Display the summary table to inspect results and to interpret findings
display(summary_table)

# Export the table to a CSV file 
summary_table.to_csv(
    OUTPUTS / "geothermal_environmental_conflict_summary.csv",
    index=False # to avoid the creation of row-number column, which is not required
)

10 Outputs

10.1 Create a static map

In [ ]:
# Convert plotted layers to Web Mercator for basemap compatibility
geothermal_web = geothermal_ni.to_crs(epsg=3857)
conflict_web = conflict_areas.to_crs(epsg=3857)

# Create figure and map axes using Matplotlib and set map size to 10 inches squared
fig, ax = plt.subplots(figsize=(10, 10))

# Plot geothermal polygons as categories clipped to Northern Ireland
geothermal_web.plot(
    ax=ax,
    column="geothermal_class", # colour polygons according to the values in 'geothermal_class', e.g., low, moderate, or high
    categorical=True, # columns treated as categories instead of continuous numeric values
    legend=True, # creates a map legend
    cmap="YlOrRd", # yellow - orange - red colour scheme represents heat
    alpha=0.5, # set transparency to 50%, i.e., semi-transparent so conflict layer is visible
    edgecolor="black", # polygon outline drawn in black
    linewidth=0.2 # sets polygon borders to a thin line
)

# Plot overlapping conflict polygons on top of geothermal polygons
conflict_web.plot(
    ax=ax,
    column="conflict_class", # colour polygons according to the values in 'conflict_class'
    categorical=True,
    legend=True,
    alpha=0.8, # Transparency set to 80% makes the overlap polygons more visible than the background geothermal polygons
    edgecolor="black",
    linewidth=0.3
)

# Add OpenStreetMap basemap
ctx.add_basemap(
    ax,
    source=ctx.providers.OpenStreetMap.Mapnik
)

# Add north arrow
add_north_arrow(ax)

# Add title
ax.set_title(
    "Potential Spatial Conflict Between Geothermal Areas and Environmentally Protected Sites in Northern Ireland",
    fontsize=13
)

# Create output file path to specify where to save the image
map_output = OUTPUTS / "geothermal_environmental_conflict_map_basemap.png"

# Save map as a PNG image with high quality print resolution and cropped excess white sapce around the map for a cleaner output
plt.savefig(map_output, dpi=300, bbox_inches="tight")

# Display map
plt.show()

# Display confirmation message that the map has been saved and its location
print("Map saved:", map_output)

10.2 Create an interactive map

10.3 Export GIS outputs

11 Conclusions and limitations